# Crawl và Chunking Health Corpus

**Mục đích**: Crawl dữ liệu từ các website y tế uy tín, chia thành chunks và lưu CSV

**Các bệnh**: cardiovascular, respiratory, diabetes, allergy

**Requirements**: trafilatura, spacy, pandas

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install trafilatura spacy
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import os
import trafilatura
import spacy
import re

# Load spacy model
print("Loading spacy model...")
nlp = spacy.load("en_core_web_sm")
print("✅ Spacy model loaded")

Loading spacy model...
✅ Spacy model loaded


In [ ]:
def clean_text(text: str) -> str:
    """Tiền xử lý text: xóa khoảng trắng thừa"""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def split_into_sentences(text: str):
    """Tách văn bản thành danh sách câu"""
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    return sentences

def chunk_sentences(sentences, chunk_size=5, overlap=2):
    """Gom n câu thành 1 chunk, overlap k câu"""
    chunks = []
    i = 0
    while i < len(sentences):
        chunk = sentences[i:i+chunk_size]
        if not chunk:
            break
        chunks.append(" ".join(chunk))
        if i + chunk_size >= len(sentences):
            break
        i += chunk_size - overlap
    return chunks

print("✅ Functions defined")

✅ Functions defined


In [29]:
# Kiểm tra môi trường và upload files
import os
import sys

print(f"Current working directory: {os.getcwd()}")
print(f"Python version: {sys.version}")

# Phát hiện môi trường
is_colab = 'google.colab' in sys.modules
print(f"Running on Colab: {is_colab}")

if is_colab:
    print("\n🔵 COLAB DETECTED - Setting up files...")

    from google.colab import files
    import zipfile
    import io

    # Check if corpus folder already exists
    if not os.path.exists('corpus') or len(os.listdir('corpus')) == 0:
        print("\n📤 Upload corpus.zip file:")
        print("   (Nén thư mục corpus/ thành corpus.zip trước khi upload)")

        uploaded = files.upload()

        if uploaded:
            for filename in uploaded.keys():
                if filename.endswith('.zip'):
                    print(f"\n📂 Extracting {filename}...")
                    with zipfile.ZipFile(io.BytesIO(uploaded[filename]), 'r') as zip_ref:
                        zip_ref.extractall('.')
                    print("✅ Extracted!")
                    break
        else:
            print("⚠️  No file uploaded!")
    else:
        print("✅ Corpus folder already exists, skipping upload")
else:
    # Local machine - chuyển vào dev_phase nếu cần
    if not os.getcwd().endswith('dev_phase'):
        if os.path.exists('dev_phase'):
            os.chdir('dev_phase')
            print(f"Changed to: {os.getcwd()}")

# Kiểm tra file CSV
csv_files = ['corpus/cardiovascular.csv', 'corpus/respiratory.csv',
             'corpus/diabetes.csv', 'corpus/allergy.csv']

print("\n📋 Checking CSV files:")
all_found = True
for csv_file in csv_files:
    if os.path.exists(csv_file):
        print(f"✅ Found: {csv_file}")
    else:
        print(f"❌ Not found: {csv_file}")
        all_found = False

if not all_found:
    print("\n⚠️  Some files missing! Please check corpus.zip structure")
    print("Expected structure:")
    print("  corpus/")
    print("    cardiovascular.csv")
    print("    respiratory.csv")
    print("    diabetes.csv")
    print("    allergy.csv")
else:
    print("\n✅ All CSV files found! Ready to crawl")


Current working directory: /content
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Running on Colab: True

🔵 COLAB DETECTED - Setting up files...

📤 Upload corpus.zip file:
   (Nén thư mục corpus/ thành corpus.zip trước khi upload)


Saving corpus.zip to corpus.zip

📂 Extracting corpus.zip...
✅ Extracted!

📋 Checking CSV files:
✅ Found: corpus/cardiovascular.csv
✅ Found: corpus/respiratory.csv
✅ Found: corpus/diabetes.csv
✅ Found: corpus/allergy.csv

✅ All CSV files found! Ready to crawl


In [30]:
# Danh sách các bệnh cần crawl
diseases = ["cardiovascular", "respiratory", "diabetes", "allergy"]

for disease in diseases:
    print(f"\n{'='*60}")
    print(f"🔄 Processing: {disease}")
    print(f"{'='*60}")

    # 1. Load CSV
    csv_file = f"corpus/{disease}.csv"
    if not os.path.exists(csv_file):
        print(f"❌ File not found: {csv_file}")
        continue

    df = pd.read_csv(csv_file)
    print(f"📋 Loaded {len(df)} URLs from {csv_file}")

    # 2. Tạo thư mục output
    output_dir = f"corpus/{disease}"
    os.makedirs(output_dir, exist_ok=True)

    # 3. Crawl từng URL
    for _, row in df.iterrows():
        source_name = row["source_name"]
        url = row["url"]

        try:
            print(f"   ↓ Crawling: {source_name}...")
            downloaded = trafilatura.fetch_url(url)

            if downloaded:
                text = trafilatura.extract(downloaded)
                if text:
                    # Clean text
                    cleaned = clean_text(text)

                    # Save to file
                    file_path = os.path.join(output_dir, f"{source_name}.txt")
                    with open(file_path, "w", encoding="utf-8") as f:
                        f.write(cleaned)
                    print(f"   ✅ Saved: {file_path}")
                else:
                    print(f"   ⚠️  Không extract được text từ {url}")
            else:
                print(f"   ❌ Không fetch được URL: {url}")
        except Exception as e:
            print(f"   ⚠️  Lỗi với {url}: {e}")

print(f"\n{'='*60}")
print("✅ CRAWL HOÀN TẤT!")
print(f"{'='*60}")


🔄 Processing: cardiovascular
📋 Loaded 7 URLs from corpus/cardiovascular.csv
   ↓ Crawling: cardiovascular_vinmec...


ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.vinmec.com/vie/bai-viet/benh-tim-mach-la-gi-nguyen-nhan-trieu-chung-va-cach-dieu-tri-vi


   ❌ Không fetch được URL: https://www.vinmec.com/vie/bai-viet/benh-tim-mach-la-gi-nguyen-nhan-trieu-chung-va-cach-dieu-tri-vi
   ↓ Crawling: cardiovascular_hellobacsi...
   ✅ Saved: corpus/cardiovascular/cardiovascular_hellobacsi.txt
   ↓ Crawling: cardiovascular_tamanhhospital...
   ✅ Saved: corpus/cardiovascular/cardiovascular_tamanhhospital.txt
   ↓ Crawling: cardiovascular_youmed...


ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.youmed.vn/tin-tuc/benh-tim-mach-nguyen-nhan-trieu-chung-va-cach-dieu-tri/


   ❌ Không fetch được URL: https://www.youmed.vn/tin-tuc/benh-tim-mach-nguyen-nhan-trieu-chung-va-cach-dieu-tri/
   ↓ Crawling: cardiovascular_benhvientim...


...

   ❌ Không fetch được URL: https://bvdk115.com.vn/di-ung-thoi-tiet-nguyen-nhan-trieu-chung-va-cach-dieu-tri.html
   ↓ Crawling: allergy_youmed...


ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.youmed.vn/tin-tuc/benh-di-ung-la-gi-nguyen-nhan-trieu-chung-cach-dieu-tri/


   ❌ Không fetch được URL: https://www.youmed.vn/tin-tuc/benh-di-ung-la-gi-nguyen-nhan-trieu-chung-cach-dieu-tri/
   ↓ Crawling: allergy_nhathuoclongchau...


ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.pharmacity.vn/bai-viet/di-ung-thoi-tiet-nguyen-nhan-trieu-chung-va-cach-phong-ngua


   ✅ Saved: corpus/allergy/allergy_nhathuoclongchau.txt
   ↓ Crawling: allergy_pharmacity...
   ❌ Không fetch được URL: https://www.pharmacity.vn/bai-viet/di-ung-thoi-tiet-nguyen-nhan-trieu-chung-va-cach-phong-ngua

✅ CRAWL HOÀN TẤT!


---
# Chunking: Chia thành các đoạn nhỏ

In [32]:
# Chunking cho từng bệnh
for disease in diseases:
    print(f"\n{'='*60}")
    print(f"✂️  Chunking: {disease}")
    print(f"{'='*60}")

    input_dir = f"corpus/{disease}"
    csv_file = f"corpus/{disease}.csv"
    output_csv = f"corpus/{disease}_chunks.csv"

    if not os.path.exists(input_dir):
        print(f"❌ Directory not found: {input_dir}")
        continue

    # Load mapping
    df_map = pd.read_csv(csv_file)
    mapping = dict(zip(df_map["source_name"], df_map["url"]))

    all_data = []

    # Process từng file txt
    for fname in os.listdir(input_dir):
        if fname.endswith(".txt"):
            source_name = os.path.splitext(fname)[0]
            url = mapping.get(source_name)

            if url is None:
                print(f"⚠️  Không tìm thấy URL mapping cho {fname}")
                continue

            try:
                with open(os.path.join(input_dir, fname), "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
            except Exception as e:
                print(f"❌ Lỗi đọc file {fname}: {e}")
                continue

            # Split into sentences
            sentences = split_into_sentences(text)
            if not sentences:
                print(f"⚠️  Không tách được câu trong {fname}")
                continue

            # Create chunks
            chunks = chunk_sentences(sentences, chunk_size=5, overlap=2)

            for c in chunks:
                all_data.append({"url": url, "chunk": c})

            print(f"   ✅ {fname}: {len(sentences)} sentences → {len(chunks)} chunks")

    # Save CSV
    if all_data:
        df_out = pd.DataFrame(all_data)
        df_out.to_csv(output_csv, index=False, encoding="utf-8")
        print(f"   💾 Saved {len(df_out)} chunks to {output_csv}")
    else:
        print(f"   ⚠️  Không có dữ liệu để lưu")

print(f"\n{'='*60}")
print("✅ CHUNKING HOÀN TẤT!")
print(f"{'='*60}")


✂️  Chunking: cardiovascular
   ✅ cardiovascular_nhathuoclongchau.txt: 35 sentences → 11 chunks
   ✅ cardiovascular_hellobacsi.txt: 102 sentences → 34 chunks
   ✅ cardiovascular_tamanhhospital.txt: 168 sentences → 56 chunks
   💾 Saved 101 chunks to corpus/cardiovascular_chunks.csv

✂️  Chunking: respiratory
   ✅ respiratory_hellobacsi.txt: 81 sentences → 27 chunks
   ✅ respiratory_nhathuoclongchau.txt: 35 sentences → 11 chunks
   💾 Saved 38 chunks to corpus/respiratory_chunks.csv

✂️  Chunking: diabetes
   ✅ diabetes_hellobacsi.txt: 148 sentences → 49 chunks
   ✅ diabetes_nhathuoclongchau.txt: 35 sentences → 11 chunks
   💾 Saved 60 chunks to corpus/diabetes_chunks.csv

✂️  Chunking: allergy
   ✅ allergy_nhathuoclongchau.txt: 35 sentences → 11 chunks
   ✅ allergy_hellobacsi.txt: 91 sentences → 30 chunks
   💾 Saved 41 chunks to corpus/allergy_chunks.csv

✅ CHUNKING HOÀN TẤT!


---
# Kiểm tra kết quả

In [33]:
# Kiểm tra các file chunks đã tạo
for disease in diseases:
    chunks_file = f"corpus/{disease}_chunks.csv"
    if os.path.exists(chunks_file):
        df = pd.read_csv(chunks_file)
        print(f"✅ {disease}_chunks.csv: {len(df)} chunks")
        print(f"   Sample chunk: {df.iloc[0]['chunk'][:100]}...")
        print()
    else:
        print(f"❌ {disease}_chunks.csv: NOT FOUND")

✅ cardiovascular_chunks.csv: 101 chunks
   Sample chunk: Ứng dụng Nhà Thuốc Long Châu Siêu ưu đãi, siêu trải nghiệm Thực phẩm chức năng Vitamin & Khoáng chất...

✅ respiratory_chunks.csv: 38 chunks
   Sample chunk: Video ngắn mới nhất Hay chóng mặt mà không biết vì sao? 🤔 96k lượt xem 30/10/2025 🌦️VÌ SAO BIẾN ĐỔI ...

✅ diabetes_chunks.csv: 60 chunks
   Sample chunk: Tiểu đường hay đái tháo đường là bệnh lý xảy ra khi lượng đường trong máu (đường huyết) tăng cao. Vậ...

✅ allergy_chunks.csv: 41 chunks
   Sample chunk: Ứng dụng Nhà Thuốc Long Châu Siêu ưu đãi, siêu trải nghiệm Thực phẩm chức năng Vitamin & Khoáng chất...

